In [ ]:
"""
TabPFN v2 Classification on CMI dataset
Predicts 'sii' using all other columns
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    accuracy_score, f1_score, log_loss
)
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# ── 1. Load data ──────────────────────────────────────────────
df = pd.read_csv("data/cmi_imputed.csv")
print(f"Dataset shape: {df.shape}")
print(f"Target 'sii' distribution:\n{df['sii'].value_counts().sort_index()}\n")

target = "sii"
X = df.drop(columns=[target])
y = df[target].values

classes = np.sort(np.unique(y))
n_classes = len(classes)
print(f"Number of classes: {n_classes} → {classes.tolist()}\n")

# ── 2. Install / import TabPFN ────────────────────────────────
# pip install tabpfn
from tabpfn import TabPFNClassifier

# ── 3. Stratified K-Fold CV ───────────────────────────────────
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_probs = np.zeros((len(y), n_classes))
oof_preds = np.zeros(len(y), dtype=int)
fold_metrics = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    clf = TabPFNClassifier(ignore_pretraining_limits=True)
    clf.fit(X_train, y_train)

    probs = clf.predict_proba(X_val)
    preds = clf.predict(X_val)

    oof_probs[val_idx] = probs
    oof_preds[val_idx] = preds

    acc = accuracy_score(y_val, preds)
    f1 = f1_score(y_val, preds, average="weighted")

    if n_classes == 2:
        auc = roc_auc_score(y_val, probs[:, 1])
    else:
        y_val_bin = label_binarize(y_val, classes=classes)
        auc = roc_auc_score(y_val_bin, probs, multi_class="ovr", average="weighted")

    fold_metrics.append({"fold": fold, "accuracy": acc, "f1_weighted": f1, "roc_auc": auc})
    print(f"Fold {fold}: Acc={acc:.4f}  F1={f1:.4f}  ROC-AUC={auc:.4f}")

# ── 4. Overall metrics ───────────────────────────────────────
print("\n" + "=" * 60)
print("OVERALL OUT-OF-FOLD RESULTS")
print("=" * 60)

overall_acc = accuracy_score(y, oof_preds)
overall_f1 = f1_score(y, oof_preds, average="weighted")

if n_classes == 2:
    overall_auc = roc_auc_score(y, oof_probs[:, 1])
else:
    y_bin = label_binarize(y, classes=classes)
    overall_auc = roc_auc_score(y_bin, oof_probs, multi_class="ovr", average="weighted")

print(f"Accuracy:       {overall_acc:.4f}")
print(f"F1 (weighted):  {overall_f1:.4f}")
print(f"ROC-AUC (ovr):  {overall_auc:.4f}")

avg = pd.DataFrame(fold_metrics).mean(numeric_only=True)
print(f"\nMean across folds → Acc={avg['accuracy']:.4f}  F1={avg['f1_weighted']:.4f}  AUC={avg['roc_auc']:.4f}")

print(f"\nClassification Report:\n")
print(classification_report(y, oof_preds, digits=4))

print("Confusion Matrix:")
print(confusion_matrix(y, oof_preds))

# ── 5. ROC Curves ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

if n_classes == 2:
    fpr, tpr, _ = roc_curve(y, oof_probs[:, 1])
    ax.plot(fpr, tpr, lw=2, label=f"ROC (AUC = {overall_auc:.4f})")
else:
    y_bin = label_binarize(y, classes=classes)
    for i, cls in enumerate(classes):
        fpr, tpr, _ = roc_curve(y_bin[:, i], oof_probs[:, i])
        cls_auc = roc_auc_score(y_bin[:, i], oof_probs[:, i])
        ax.plot(fpr, tpr, lw=2, label=f"Class {cls} (AUC = {cls_auc:.4f})")

ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("TabPFN v2 — ROC Curves (5-Fold CV)")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("tabpfn_roc_curves.png", dpi=150)
print("\nROC curve saved to tabpfn_roc_curves.png")

Dataset shape: (8243, 47)
Target 'sii' distribution:
sii
0.0    5696
1.0    1547
2.0     914
3.0      86
Name: count, dtype: int64

Number of classes: 4 → [0.0, 1.0, 2.0, 3.0]

